In [1]:
%pip install torch torchvision opencv-python-headless numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class CrystalStepperDataset(Dataset):
    def __init__(self, video_dir, chunk_size=256, max_frames=30, start_second=40):
        self.video_dir = video_dir
        self.chunk_size = chunk_size
        self.max_frames = max_frames
        self.start_second = start_second
        self.samples = []
        
        self._process_videos()

    def _process_videos(self):
        video_files = [f for f in os.listdir(self.video_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
        
        for vid_file in video_files:
            vid_path = os.path.join(self.video_dir, vid_file)
            cap = cv2.VideoCapture(vid_path)
            
            fps = cap.get(cv2.CAP_PROP_FPS)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if fps <= 0: fps = 30 
                
            start_frame_idx = int(self.start_second * fps)
            if start_frame_idx >= total_frames:
                cap.release()
                continue
                
            frame_indices = np.linspace(start_frame_idx, total_frames - 1, self.max_frames, dtype=int)
            
            frames = []
            for idx in frame_indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
                ret, frame = cap.read()
                if not ret: break
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                gray = cv2.resize(gray, (1024, 768))
                frames.append(gray)
            cap.release()
            
            if len(frames) < self.max_frames: continue
            video_np = np.array(frames, dtype=np.float32)
            
            # Global min-max normalize
            v_min, v_max = video_np.min(), video_np.max()
            if v_max > v_min:
                video_np = (video_np - v_min) / (v_max - v_min)
            
            T, H, W = video_np.shape
            
            # Slice into 256x256 chunks
            for i in range(0, H, self.chunk_size):
                for j in range(0, W, self.chunk_size):
                    chunk = video_np[:, i:i+self.chunk_size, j:j+self.chunk_size]
                    
                    if chunk.shape[1] == self.chunk_size and chunk.shape[2] == self.chunk_size:
                        if np.var(chunk, axis=0).mean() < 0.0005: continue 
                        
                        # CREATE STEPPING PAIRS: Input (t) -> Target (t+1)
                        for t in range(T - 1):
                            frame_t = chunk[t]
                            target_t = chunk[t+1] 
                            
                            param_channel = np.ones_like(frame_t)
                            x = np.linspace(0, 1, self.chunk_size)
                            y = np.linspace(0, 1, self.chunk_size)
                            X, Y = np.meshgrid(x, y)
                            
                            input_tensor = np.stack([frame_t, param_channel, X, Y], axis=0)
                            self.samples.append((input_tensor, target_t))
                            
        print(f"Generated {len(self.samples)} consecutive step-pairs for training.")

    # FIX: These must be perfectly left-aligned with "def _process_videos(self):"
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        X, y = self.samples[idx]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Instantiate the dataset
dataset = CrystalStepperDataset(video_dir='Riboflavin_Videos', chunk_size=256, max_frames=30, start_second=40)

# Train/Test Split
train_size = int(0.90 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

Generated 696 consecutive step-pairs for training.


In [10]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """(convolution => [BN] => LeakyReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNetStepper(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()
        # Downsampling path
        self.inc = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down1 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(128, 256)
        
        # Upsampling path with skip connections
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(256, 128) # 128 (from up) + 128 (from skip)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(128, 64)   # 64 (from up) + 64 (from skip)
        
        # Final projection layer to map back to a single grayscale frame
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.pool1(x1)
        x3 = self.down1(x2)
        x4 = self.pool2(x3)
        x5 = self.down2(x4)
        
        # Decoder with Skip-Connections tracking local features
        out = self.up1(x5)
        out = torch.cat([out, x3], dim=1)
        out = self.conv_up1(out)
        
        out = self.up2(out)
        out = torch.cat([out, x1], dim=1)
        out = self.conv_up2(out)
        
        logits = self.outc(out)
        return logits.squeeze(1) # Shape: (Batch, H, W)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNetStepper(in_channels=4, out_channels=1).to(device)
print(f"U-Net Front-Tracking Stepper successfully deployed on: {device}")

U-Net Front-Tracking Stepper successfully deployed on: cuda


In [11]:
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

epochs = 30
criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print("Training Localized U-Net Front Stepper...")
for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        predictions = torch.clamp(predictions, 0.0, 1.0)
        
        # Spatial loss tracking local crystal shapes
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        
    scheduler.step()
    train_loss /= len(train_loader.dataset)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{epochs}] — Localized Front Loss: {train_loss:.6f}")

print("U-Net training pass completed.")

Training Localized U-Net Front Stepper...
Epoch [1/30] — Localized Front Loss: 0.018753
Epoch [5/30] — Localized Front Loss: 0.000828
Epoch [10/30] — Localized Front Loss: 0.000604
Epoch [15/30] — Localized Front Loss: 0.000479
Epoch [20/30] — Localized Front Loss: 0.000498
Epoch [25/30] — Localized Front Loss: 0.000364
Epoch [30/30] — Localized Front Loss: 0.000306
U-Net training pass completed.


In [9]:
from IPython.display import HTML, display

model.eval()
os.makedirs('FNO_Outputs', exist_ok=True)

with torch.no_grad():
    for idx, (test_X, test_y) in enumerate(test_loader):
        # We will loop and generate a 30-frame video out of thin air step-by-step
        generated_frames = []
        ground_truth_frames = []
        
        # Pull initial frame from our test sample
        current_input = test_X.to(device) # Shape: (1, 4, 256, 256)
        
        # Let's roll forward for 25 steps
        for step in range(25):
            # 1. Predict next frame
            pred_frame = model(current_input) # Shape: (1, 256, 256)
            pred_frame = torch.clamp(pred_frame, 0.0, 1.0)
            
            pred_np = pred_frame.cpu().numpy()[0]
            generated_frames.append((pred_np * 255.0).astype(np.uint8))
            
            # 2. Update the input tensor for the NEXT step using the prediction we just made!
            next_input = current_input.clone()
            next_input[0, 0] = pred_frame[0] # Inject prediction into the Frame channel
            current_input = next_input
            
        # Compile generated frames into a video file
        pred_video_path = f'FNO_Outputs/autoregressive_prediction_{idx}.mp4'
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(pred_video_path, fourcc, 5, (256, 256), isColor=False)
        for frame in generated_frames:
            out.write(frame)
        out.release()
        
        print("Autoregressive simulation complete. Rendering physical rollout:")
        
        html_player = f"""
        <div style="background-color: #1e1e1e; padding: 15px; border-radius: 8px; text-align: center; width: 350px; margin: 0 auto;">
            <h3 style="color: white; margin-bottom: 10px;">Autoregressive FNO Rollout</h3>
            <video width="320" height="320" controls autoplay loop muted src="{pred_video_path}" type="video/mp4">
            </video>
        </div>
        """
        display(HTML(html_player))
        break

Autoregressive simulation complete. Rendering physical rollout:
